# TD — Tropical Cyclones and Mortality
## Socioeconomic Impacts of Natural Disasters

**Guiding question:** Are the most powerful cyclones the deadliest? Or does income explain what wind speed cannot?

| Dataset | Description |
|---|---|
| `ibtracs_storms.csv` / `ibtracs_storms_land.csv` | IBTrACS global cyclone tracks — preprocessed |
| `emdat_tropical_cyclones_preprocessed.csv` | EM-DAT tropical cyclone events — preprocessed |
| `country_gdp_income.csv` | World Bank income groups and GDP |


---
## Part 0 — Setting Up



### 0.1 — Import libraries

To start, import the core libraries : pandas, numpy, matplotlib, and geopandas. Also set some display options for better readability.


In [13]:
...

Ellipsis

### 0.2 — Load Natural Earth coastlines

Load land polygons with `gpd.read_file(WORLD_URL)`. These will be drawn on every map with:
```python
fig, ax = plt.subplots(figsize=(12, 6))
world.plot(ax=ax, color='#e8e8e8', edgecolor='white', linewidth=0.4, zorder=1)
ax.set_facecolor('#c6e0f5')
```


In [14]:
# Natural Earth 110m land polygons — used for all maps
import geopandas as gpd
WORLD_URL = ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector"
             "/master/geojson/ne_110m_land.geojson")
world = gpd.read_file(WORLD_URL)
print("World land polygons loaded:", world.shape)


World land polygons loaded: (127, 4)


### 0.3 - Load preprocessed IBTrACS

The two datasets were built in `utils.ipynb` and saved to `data_final/`:
- `storms` - one row per storm at its **global peak**, meaning the maximal speed it reached anywhere in its life cycle
- `storms_land` - one row per storm at its **near-land peak** (used for mortality analysis)

Both have a `NAME_NORM` column formatted as `STORMNAME_YEAR_MONTH` (e.g. `KATRINA_2005_08`).

Load it with `pd.read_csv()` and inspect the columns. The key variables are:
- `NAME_NORM` - storm identifier
- `USA_WIND` - maximum wind speed in knot (need to multiply by 1.852 to get km/h)
- `YEAR` - year of the storm
- `MONTH` - month of the storm
- `LATITUDE` and `LONGITUDE` - location of the storm at its peak (global or near-land)
- `BASIN` - basin where the storm occurred (e.g. North Atlantic, Western Pacific, etc.)



In [15]:

storms_land = ...
storms = ...


In [16]:
# IBTrACS preprocessing was performed in utils.ipynb and saved to data_final/.
# No computation needed here.


### 0.4 — Load preprocessed EM-DAT

Load `data_final/emdat_tropical_cyclones_preprocessed.csv`. It already contains a `NAME_NORM` column matching the IBTrACS format for events where the storm name was identified.

Load it and explore the columns.


In [17]:
em = ...


### 0.5 — Load country metadata and join to EM-DAT

Load `data_final/country_gdp_income.csv`. It contains country metadata from the World Bank, including income group and GDP.

Merge on `ISO` (EM-DAT) = `iso3` (meta).


In [18]:
gdp = ...

### 0.6 — Shared colour palettes

In [19]:
BASIN_COLORS = {
    "North Atlantic":  "#e41a1c",
    "Western Pacific": "#377eb8",
    "Eastern Pacific": "#ff7f00",
    "North Indian":    "#984ea3",
    "South Indian":    "#4daf4a",
    "South Pacific":   "#a65628",
    "South Atlantic":  "#999999",
}
ORDER = ["Low income", "Lower-middle income", "Upper-middle income", "High income"]
COLORS = {
    "Low income":          "#d73027",
    "Lower-middle income": "#fc8d59",
    "Upper-middle income": "#91bfdb",
    "High income":         "#4575b4",
}


---
## Part 1 — IBTrACS: The Physical Record

### 1.1 — How many tropical cyclones have been recorded since 1980?

Print `len(storms)` and `storms['BASIN'].value_counts()`.


In [20]:
# 1.1 — Count storms and breakdown by basin
...

### 1.2 — Where are tropical cyclones located?

Plot the 50 most intense storms (`storms.nlargest(50, 'USA_WIND')`), one point per storm, coloured by basin.

You can use a loop for each basin to plot them separately.

_Hint — map skeleton:_
```python
fig, ax = plt.subplots(figsize=(15, 7))
world.plot(ax=ax, color='#e8e8e8', edgecolor='white', linewidth=0.4, zorder=1)
ax.set_facecolor('#c6e0f5')
# then for each basin: ax.scatter(grp['LON'], grp['LAT'], color=BASIN_COLORS[basin], ...)
ax.set_xlim(-180, 180); ax.set_ylim(-60, 60)
```


In [21]:
# 1.2 — Map of the 50 most intense TCs
...

### 1.3 — The most extreme cyclones

Show the 15 most intense storms in a horizontal bar chart (`storms.nlargest(15, 'USA_WIND')`), coloured by basin.


In [22]:
# 1.3 — Bar chart: 15 most intense TCs
...

---
## Part 2 — EM-DAT: The Impact Record

### 2.1 — How many tropical cyclone events are recorded in EM-DAT?



In [23]:
# 2.1 — Count EM-DAT events
...

### 2.2 — Mean and median deaths, affected persons, and damages per event

Why such a difference between mean and median? What does it tell us about the distribution of impacts?


In [24]:
# 2.2 — Summary statistics for impact variables
...

### 2.3 — Most deadly and most destructive events

Draw two side-by-side horizontal bar charts: the 15 deadliest events and the 15 most destructive by damage. Colour bars by `income_group` using `COLORS`.

_Hint — bar chart skeleton:_
```python
top = em_deaths.nlargest(15, 'Total Deaths').sort_values('Total Deaths')
bar_colors = [COLORS.get(g, '#cccccc') for g in top['income_group']]
ax.barh(top['label'], top['Total Deaths'] / 1000, color=bar_colors)
```

**Q2.3 —** Why do rich countries dominate the damages ranking but not the deaths ranking?


In [25]:
# 2.3 — Bar charts: deadliest and most destructive
...

---
## Part 3 — Coupling IBTrACS and EM-DAT

### 3.1 — How many storms are matched?

Both `storms_land` and `em` have a `NAME_NORM` column. Merge them on this key:
```python
matched = storms_land.merge(
    em[['NAME_NORM', 'Total Deaths', 'Total Affected', "Total Damage ('000 US$)"]],
    on='NAME_NORM'
)
```
Print the number of matched rows and the match rate vs. `len(storms_land)`.

**Q3.1 —** How many storms appear in the merged dataset? Why might some storms not match?

In [26]:
# 3.1 — Match IBTrACS to EM-DAT
...

### 3.2 — What was Hurricane Katrina's maximum wind speed?

Filter `storms_land` for Katrina and Nargis:
```python
katrina = storms_land[(storms_land['NAME'] == 'KATRINA') & (storms_land['SEASON'] == 2005)]
nargis  = storms_land[(storms_land['NAME'] == 'NARGIS')  & (storms_land['SEASON'] == 2008)]
```
Print the `USA_WIND` values (knots) and convert to km/h (× 1.852).

**Q3.2 —** Katrina was stronger than Nargis at landfall yet killed 75 times fewer people. What does this tell us about wind speed as a predictor of mortality?


In [27]:
# 3.2 — Katrina and Nargis wind speed lookup
...

### 3.3 — Correlation: wind speed vs. deaths, affected, and damages

For each impact variable, compute the Pearson correlation with near-land peak wind speed, using a log transformation on the impact variable:
```python
r = matched['USA_WIND'].corr(np.log1p(matched['Total Deaths']))
```
Then produce a 3-panel scatter plot (one panel per variable), with `r` annotated on each panel.

**Q3.3 —** Which correlation is strongest? Which is weakest? Why does damage correlate more strongly with wind speed than deaths do?


In [28]:
# 3.3 — Correlation scatter plots
...

---
## Part 4 — The Role of Income

### 4.1 — 15 most deadly and most destructive: who is hit?

Draw the same side-by-side charts as in 2.3, but for all EM-DAT events (not limited to the matched subset), coloured by income group.

**Q4.1 —** The two lists barely overlap. What does this reversal tell us about how income shapes cyclone impact?


In [29]:
# 4.1 — Side-by-side: deadliest vs. most destructive
...

### 4.2 — GDP per capita vs. three impact dimensions

For each country, compute the average of each impact variable across events with positive values. Then plot GDP per capita (x-axis, log scale) against each average (y-axis, log scale) in a 3-panel figure, coloured by income group.

_Hint — group and plot:_
```python
avg = (em.dropna(subset=['gdp_pc_usd'])
         .groupby(['Country', 'income_group', 'gdp_pc_usd'])
         .agg(avg_deaths=('Total Deaths', lambda x: x[x>0].mean()))
         .reset_index())
ax.set_xscale('log'); ax.set_yscale('log')
ax.scatter(avg['gdp_pc_usd'], avg['avg_deaths'], color=COLORS[group])
```

**Q4.2 —** The damage panel shows a positive slope — richer countries record more damage per event. Is this paradoxical? Name two mechanisms that explain it.


In [30]:
# 4.2 — GDP/cap vs. impact dimensions (3-panel scatter)
...

---
## Part 5 — Temporal Comparison: 2005–2014 vs. 2015–2024

### 5.1–5.3 — Summary table

Split the EM-DAT data into two periods and compare:
```python
p1 = em[(em['Start Year'] >= 2005) & (em['Start Year'] <= 2014)]
p2 = em[(em['Start Year'] >= 2015) & (em['Start Year'] <= 2024)]
```
Print total deaths, total damage (bn USD), and event counts for each period.

**Q5.1 —** If deaths per event declined between the two periods, what structural changes could explain this improvement?


In [31]:
# 5.1-5.3 — Summary table
...

### Visual comparison

Draw three bar charts side-by-side: annual deaths, annual damages, and annual storm counts (from `storms`). Shade the two periods in different background colours:
```python
ax.axvspan(2005, 2014.5, alpha=0.12, color='steelblue', label='2005–2014')
ax.axvspan(2014.5, 2024, alpha=0.12, color='darkorange', label='2015–2024')
```


In [32]:
# Visual comparison: deaths / damages / storm counts
...

---
## Part 6 — Discussion

**Q6.1 —** Based on Parts 2–4, characterise the inequality of cyclone impact across income groups. Is it primarily an inequality of **exposure** (physical) or **vulnerability** (institutional)? Support your answer with at least two observations from the data.

**Q6.2 —** If you were advising an international fund allocating reconstruction and adaptation aid, what conclusions or recommendations would the evidence in this TD support?
